In [1]:
import os, glob
import pandas as pd
from collections import defaultdict
import numpy as np
from tqdm import tqdm

In [2]:
with open('/Users/ginoprasad/Job_Applications/web_crawler/aladdin/dataset/diff.txt') as infile:
    diffs = [x.strip() for x in infile.read().split('-'*50) if x.strip()]

In [3]:
df = defaultdict(list)
for diff in diffs:
    html_files, *element_diffs = diff.split('\n\n')
    html_file_1, html_file_2 = html_files.split('\n')
    page_num = '_'.join(os.path.basename(html_file_1).split('_')[:2])
    dirname = os.path.basename(os.path.dirname(os.path.dirname(html_file_1)))
    for element_diff in element_diffs:
        df['dirname'].append(dirname)
        df['page'].append(page_num)
        df['html_file_1'].append(html_file_1)
        df['html_file_2'].append(html_file_2)
        location, element_1, element_2 = element_diff.split('\n')
        df['location'].append(location)
        df['element_1'].append(element_1)
        df['element_2'].append(element_2)
df = pd.DataFrame(df)
df

,dirname,page,html_file_1,html_file_2,location,element_1,element_2
0,Stryker_v1.3,page_1,/Users/ginoprasad/Job_Applications/web_crawler...,/Users/ginoprasad/Job_Applications/web_crawler...,"-,315",-,"<style data-emotion=""css"" data-s=""""></style>"
1,Stryker_v1.3,page_1,/Users/ginoprasad/Job_Applications/web_crawler...,/Users/ginoprasad/Job_Applications/web_crawler...,"316,317","<style data-emotion=""css"" data-s=""""></style></...","<style id=""detectElementResize"" type=""text/css..."
2,Stryker_v1.3,page_1,/Users/ginoprasad/Job_Applications/web_crawler...,/Users/ginoprasad/Job_Applications/web_crawler...,"317,318","<body id=""vpsBody"" class=""WELP WFMP highContra...","<body id=""vpsBody"" class=""WELP WFMP highContra..."
3,Stryker_v1.3,page_1,/Users/ginoprasad/Job_Applications/web_crawler...,/Users/ginoprasad/Job_Applications/web_crawler...,"2761,2762","<button aria-haspopup=""listbox"" type=""button"" ...","<button aria-haspopup=""listbox"" type=""button"" ..."
4,Stryker_v1.3,page_1,/Users/ginoprasad/Job_Applications/web_crawler...,/Users/ginoprasad/Job_Applications/web_crawler...,"2762,2763","<input type=""text"" class=""css-77hcv"" value="""">","<input type=""text"" class=""css-77hcv"" value=""b3..."
...,...,...,...,...,...,...,...
884,broadcom_app_v1.1,page_5,/Users/ginoprasad/Job_Applications/web_crawler...,/Users/ginoprasad/Job_Applications/web_crawler...,"-,2878",-,"<path d=""M10.34 14.043l5.101-5.936a.287.287 0 ..."
885,broadcom_app_v1.1,page_5,/Users/ginoprasad/Job_Applications/web_crawler...,/Users/ginoprasad/Job_Applications/web_crawler...,"2879,2883","<div class=""css-d3pjdr"">","<div class=""css-d3pjdr"" disabled="""">"
886,broadcom_app_v1.1,page_5,/Users/ginoprasad/Job_Applications/web_crawler...,/Users/ginoprasad/Job_Applications/web_crawler...,"2880,2884","<input id=""64cbff5f364f10000af3af293a050000"" t...","<input id=""64cbff5f364f10000af3af293a050000"" t..."
887,broadcom_app_v1.1,page_5,/Users/ginoprasad/Job_Applications/web_crawler...,/Users/ginoprasad/Job_Applications/web_crawler...,"2882,2886","<div class=""css-1ikf28c"">","<div class=""css-1ikf28c"" disabled="""">"


In [102]:
valid = np.vectorize(lambda elem: any([elem.startswith(f'<{x}') for x in ('input', 'button', 'textarea')]))
filtered_df = df[((valid(df['element_1']) + valid(df['element_2'])) * (df['element_1'] != '-') * (df['element_2'] != '-')).astype(bool)].reset_index()
del filtered_df['index']

In [105]:
def format_html(a):
    return a.replace('\n', '').replace('<', '\n<').replace('"\n<', '"<').replace('\n</', '</').strip().split('\n')

In [137]:
get_input_elems = np.vectorize(lambda elem: any([elem.startswith(f'<{x}') for x in ('input', 'button', 'textarea')]))
valid_input_elements = {x: y for x, y in zip(filtered_df['element_1'], filtered_df['element_2'])}

In [139]:
with open('dataset/excluded_f1.txt', 'w') as excluded_outfile:
    with open('dataset/included_f1.txt', 'w') as included_outfile:
        with open('dataset/included_f1_key.txt', 'w') as included_key_outfile:
            for html_file in tqdm(filtered_df.html_file_1.unique()):
                with open(html_file) as infile:
                    s = format_html(infile.read())
                    input_elems = np.array(s)[get_input_elems(s)]

                    for input_elem in input_elems:
                        if input_elem in valid_input_elements_set:
                            included_outfile.write(f"{input_elem}\n")
                            included_key_outfile.write(f"{valid_input_elements[input_elem]}\n")
                        else:
                            excluded_outfile.write(f"{input_elem}\n")

100%|███████████████████████████| 12/12 [00:10<00:00,  1.18it/s]
